# ScyPibanya — Erde-Mond-Simulation + Gun Club

Gruppe: Nico Hees, Ahmad Alkhaddour, Bastian Klumpp, Youssef Fahmy, Jan Schäfer<br>
DHBW Stuttgart, Scientific Programming Lab

In [ ]:
#imports

import sys
sys.path.insert(0, "src")
sys.path.insert(1, "tests")

from panel import run, render, build_panel
from analysis import sweep_fast, plot_corridor_heatmap

import ipywidgets as widgets
from IPython.display import HTML
import numpy as np
import unittest

## Projektüberblick & Architektur

Das Projekt ist in kleine, klar getrennte Module (`src/`) aufgeteilt — jedes hat eine Aufgabe.
Das Notebook importiert nur die obersten Funktionen; die Arbeit passiert in diesen Modulen.

| Modul | Aufgabe |
|---|---|
| `constants.py` | Physikalische Konstanten & Startwerte (SI: m, kg, s) |
| `body.py` | Ein Körper: Masse, Radius, Position, Geschwindigkeit + Kinematik |
| `integrator.py` | Gravitationskraft + Zeitschritt-Verfahren (Störmer-Verlet, Euler) |
| `collisions.py` | Berührung erkennen + inelastisch verschmelzen |
| `scenarios.py` | Fertige Startaufstellungen (Erde-Mond, Kanonenschuss) |
| `simulation.py` | Motor: führt die Zeitschritte aus, sammelt die History |
| `visualization.py` | History → matplotlib-Animation |
| `analysis.py` | Parameter-Sweep + Trefferauswertung (Heatmap) |
| `panel.py` | Bedien-Panel (ipywidgets) + `run` / `render` fürs Notebook |

**Datenfluss:**

```
scenarios          simulation           integrator                 visualization / analysis
create_*()   ──▶   build + simulate ──▶ Kraft + Verlet-Schritt ──▶ History ──▶ Animation / Heatmap
```

Jeder Lauf ist dieselbe Kette: ein Szenario liefert die Startkörper → die Simulation
schrittet sie durch die Zeit → in jedem Schritt berechnet der Integrator Gravitation und neue
Position → Kollisionen werden geprüft → jeder Schritt wird als Snapshot in der History
abgelegt. Aus der History entsteht am Ende eine Animation oder die Trefferauswertung.

**Konventionen & Qualität:** SI-Einheiten (m/kg/s), englische Bezeichner, eine Testdatei pro Modul
(`tests/`), Git mit Branch-Protection + CI-Checks. Am Ende laufen alle Tests automatisch.

## 1. Mathemathische und physikalische Grundlagen

## Definition 1.1 — Newtons Gravitationsgesetz

Zwei Punktmassen $m_1$ und $m_2$ im Abstand $r$ ziehen sich mit der Kraft

$$F = G \cdot \frac{m_1 \cdot m_2}{r^2}$$

an, wobei

$$G = 6.67430 \times 10^{-11}\ \text{m}^3\text{kg}^{-1}\text{s}^{-2}$$

die Gravitationskonstante ist. Die Kraft wirkt entlang der Verbindungslinie beider Körper und ist stets anziehend.

---

## Satz 1.2 — Bewegungsgleichung (Mehrkörperproblem)

Aus Newtons zweitem Axiom $F = m \cdot a$ und Definition 1.1 folgt für die Gesamtbeschleunigung eines Körpers $i$ unter dem Einfluss aller anderen Körper $j$:

$$\vec{a}_i = \sum_{j \neq i} G \cdot \frac{m_j}{r_{ij}^2} \cdot \hat{r}_{ij}$$

Für mehr als zwei Körper besitzt dieses gekoppelte Differentialgleichungssystem keine geschlossene analytische Lösung (**Mehrkörperproblem**). Wir lösen es deshalb numerisch (siehe Definition 1.3).

> **Numerische Absicherung:** Um eine Division durch Null bei extrem nahen Körpern zu vermeiden, wird die paarweise Kraftberechnung nur für $r > 10^{-12}\,\text{m}$ ausgewertet — ohne physikalische Bedeutung, rein numerisch.

---

## Definition 1.3 — Numerische Zeitintegration

Da die Bewegungsgleichung aus Satz 1.2 nicht analytisch lösbar ist, nähern wir uns der Bahn in diskreten Zeitschritten $\Delta t$. Wir implementieren zwei Verfahren mit unterschiedlicher Genauigkeit an:

**Explizites Euler-Verfahren**

Geschwindigkeit und Position werden direkt aus dem alten Zustand fortgeschrieben:

$$\vec{v}_{n+1} = \vec{v}_n + \vec{a}_n \cdot \Delta t \qquad \vec{x}_{n+1} = \vec{x}_n + \vec{v}_n \cdot \Delta t$$

Einfach und schnell, akkumuliert aber pro Schritt einen Fehler erster Ordnung — bei langen Simulationszeiten (z. B. mehrere Mondumläufe) wächst dieser Fehler spürbar an.

**Störmer-Verlet-Verfahren**

Statt der Geschwindigkeit wird die vorherige Position $\vec{x}_{n-1}$ gespeichert, und die neue Position direkt aus den letzten beiden Positionen plus Beschleunigung berechnet:

$$\vec{x}_{n+1} = 2\vec{x}_n - \vec{x}_{n-1} + \vec{a}_n \cdot \Delta t^2$$

Die Geschwindigkeit wird nachträglich per zentralem Differenzenquotient geschätzt:

$$\vec{v}_n = \frac{\vec{x}_{n+1} - \vec{x}_{n-1}}{2\Delta t}$$

Da beim allerersten Schritt noch keine Vorgänger-Position existiert, wird sie einmalig per Euler-Rückwärtsschritt initialisiert ($\vec{x}_{-1} = \vec{x}_0 - \vec{v}_0 \cdot \Delta t$).

---

## Definition 1.4 — Fluchtgeschwindigkeit

Die Fluchtgeschwindigkeit $v_{esc}$ ist die minimale Startgeschwindigkeit, mit der ein Körper das Gravitationsfeld einer Masse $M$ mit Radius $R$ vollständig verlassen kann:

$$v_{esc} = \sqrt{\frac{2GM}{R}}$$

Für die Erde ergibt sich mit `EARTH_MASS` und `EARTH_RADIUS`:

$$v_{esc} \approx 11{,}19\ \text{km/s} \quad (\texttt{constants.EARTH\_ESCAPE\_VELOCITY})$$

Dieser Wert markiert die harte untere Geschwindigkeitsgrenze unseres Trefferkorridors (Abschnitt 1.7): unterhalb dieser Schwelle fällt jedes Geschoss unabhängig vom Abschusswinkel zur Erde zurück.

---

## Definition 1.5 — Kreisbahngeschwindigkeit

Damit ein Körper im Abstand $r$ um eine Zentralmasse $M$ auf stabiler Kreisbahn bleibt, muss die Gravitationskraft genau der Zentripetalkraft entsprechen:

$$G\frac{Mm}{r^2} = \frac{mv^2}{r} \quad \Rightarrow \quad v_{circ} = \sqrt{\frac{GM}{r}}$$

Mit `EARTH_MASS` und `EARTH_MOON_DISTANCE` ergibt sich:

$$v_{circ} \approx 1018\ \text{m/s} \quad (\texttt{constants.MOON\_CIRCULAR\_VELOCITY})$$

— genau die Startgeschwindigkeit, die der Mond in unserem Erde-Mond-Szenario in +Y-Richtung erhält, während die Erde im Ursprung ruht.

---

## Definition 1.6 — Vollständig inelastische Kollision

Berühren sich zwei Körper, gehen wir laut Aufgabenstellung von einer 100 % inelastischen Kollision aus: Beide Körper verschmelzen zu einem, Masse und Impuls bleiben erhalten:

$$m_{ges} = m_1 + m_2 \qquad \vec{v}_{neu} = \frac{m_1\vec{v}_1 + m_2\vec{v}_2}{m_1+m_2}$$

Ein Treffer des Mondes äußert sich im Simulationsverlauf dadurch, dass ein neuer, verschmolzener Körper auftaucht und einer der ursprünglichen Namen (Mond, Geschoss) verschwindet.

---

## 1.7 Der Trefferkorridor

Ein erster grober Test (Abschnitt 4, drei Geschwindigkeiten × drei Winkel, 1 Tag Simulationsdauer) zeigte: Bei 7 und 9 km/s — beide unterhalb der Fluchtgeschwindigkeit aus Definition 1.4 (≈ 11.19 km/s) — fällt das Geschoss unabhängig vom Winkel zur Erde zurück. Bei 12 km/s reicht die Geschwindigkeit zwar aus, um die Erde zu verlassen, aber keiner der drei groben Winkel (0°, 15°, 30°) trifft den Mond.

Das deutet bereits an, dass ein Treffer nicht an einem einzelnen Parameterpaar hängt, sondern nur in einem schmalen Bereich möglich ist. Um diesen Bereich sichtbar zu machen, haben wir mit `sweep_fast()` ein deutlich feineres Raster ausgewertet:

- **51 Geschwindigkeiten:** 10.5–13.0 km/s, Schritt 50 m/s
- **41 Winkel:** 2°–22°, Schritt 0.5°
- **≈ 2000 Zellen**, ausgewertet in wenigen Sekunden dank vektorisierter Verlet-Propagation durch eine einmal vorberechnete Erde-Mond-Bahn

Das Ergebnis (`plot_corridor_heatmap`) bestätigt die Vermutung: Es gibt keinen isolierten Trefferpunkt, sondern einen zusammenhängenden **Trefferkorridor** — ein schmales, diagonal verlaufendes Band im Geschwindigkeit-Winkel-Raum.

Je höher die Startgeschwindigkeit, desto kleiner der nötige Vorhaltewinkel (von rund 21° bei 11.3 km/s bis auf rund 8–9° bei 13 km/s). Das ist physikalisch plausibel — ein schnelleres Geschoss braucht weniger Flugzeit, wodurch der Mond auf seiner Umlaufbahn weniger Zeit hat, sich weiterzubewegen, und ein kleinerer Vorhalt genügt.

Nach links ist der Korridor durch eine harte Wand begrenzt: Die weiße gestrichelte Linie bei $v_{esc} \approx 11.19$ km/s markiert exakt die Fluchtgeschwindigkeit aus Definition 1.4. Links davon bleibt die Farbe der Heatmap durchgehend dunkel (= große Distanz zum Mond) — kein einziger Treffer, egal bei welchem Winkel.

Das erklärt auch rückblickend, warum die groben Testwerte (7, 9, 12 km/s × 0°, 15°, 30°) keinen Treffer erzielten: 7 und 9 km/s lagen unter der Wand, und bei 12 km/s waren alle drei getesteten Winkel zu weit vom schmalen Korridor entfernt, der bei dieser Geschwindigkeit nur etwa 11–12° breit ist.



### Satz 1.2 im Code — Gravitationsbeschleunigung

`Integrator.calculate_acceleration` summiert für jeden Körper die Beschleunigung durch alle
anderen Körper — genau die Summe aus Satz 1.2.

```python
# src/integrator.py
def calculate_acceleration(self, bodies):
    for i in bodies:
        i.acceleration = np.zeros(3)
    for i in range(len(bodies)):
        for j in range(i + 1, len(bodies)):

            p1 = bodies[i]
            p2 = bodies[j]
            direction = Body.distance_vector_to(p1, p2)
            r = np.linalg.norm(direction)
            if r > 1e-12:
                direction_norm = direction / r

                F = G * p1.mass * p2.mass / r**2

                a1 = F / p1.mass
                a2 = F / p2.mass

                bodies[i].acceleration += a1 * direction_norm
                bodies[j].acceleration -= a2 * direction_norm
```

- **Reset:** zu Beginn jedes Schritts wird die Beschleunigung auf Null gesetzt, dann frisch aufsummiert.
- **Jedes Paar genau einmal:** `j = i + 1` — das ist die Summe aus Satz 1.2.
- **Newton wörtlich:** `F = G · m₁ · m₂ / r²` (Definition 1.1).
- **actio = reactio:** ein Körper `+=`, der andere `-=` — Newtons 3. Axiom, ohne die Kraft doppelt zu rechnen.
- **Numerik-Schutz:** `r > 1e-12` verhindert Division durch Null (rein numerisch).

### Definition 1.3 im Code — der Störmer-Verlet-Schritt

`Verlet.step` ist die positionsbasierte Zeitintegration aus Definition 1.3.

```python
# src/integrator.py
class Verlet(Integrator):
    def step(self, bodies, dt):

        Integrator.calculate_acceleration(self, bodies)

        # Mini-Euler only at the first step:
        # sets position_previous backwards, without moving the position
        for body in bodies:
            if body.position_previous is None:
                body.position_previous = body.position - body.velocity * dt

        for i in range(len(bodies)):
            prev_pos = bodies[i].position_previous.copy()
            temp_pos = bodies[i].position.copy()

            bodies[i].position = bodies[i].position * 2 - prev_pos + bodies[i].acceleration.copy() * dt * dt
            bodies[i].position_previous = temp_pos

            # Calculate velocity implicitly from position change
            bodies[i].velocity = (bodies[i].position.copy() - prev_pos) / (2.0 * dt)
```

- Erst die Kräfte (`calculate_acceleration`), dann bewegen.
- **Erster Schritt:** es gibt noch keine Vorgänger-Position → einmalig `x₋₁ = x₀ − v₀·dt` (Euler-Rückwärtsschritt).
- **Kernschritt** = Verlet-Formel: `x_{n+1} = 2·x_n − x_{n-1} + a·dt²`.
- Geschwindigkeit per zentralem Differenzenquotient `v_n = (x_{n+1} − x_{n-1}) / (2·dt)`.

**Zum Vergleich — der einfache Euler** (Definition 1.3, „einfach aber ungenau"):

```python
# src/integrator.py
class Euler(Integrator):
    def step(self, bodies, dt):
        # explicit euler with big errors
        Integrator.calculate_acceleration(self, bodies)

        for body in bodies:
            temp_vel = body.velocity.copy()

            body.velocity += body.acceleration * dt
            body.position += temp_vel * dt
```

Euler schreibt Geschwindigkeit und Position direkt aus dem aktuellen Zustand fort — Fehler erster
Ordnung, der sich bei langen Läufen aufschaukelt. Deshalb ist unser Standard Verlet.

## 2. Teil 1: Erde-Mond-System

### Aufbau der Simulation

Der Erdmittelpunkt liegt im Koordinatenursprung $(0,0,0)$, wie in der Aufgabenstellung
vorgegeben. Der Mond startet auf der positiven X-Achse im Abstand `EARTH_MOON_DISTANCE`
und bewegt sich in +Y-Richtung mit der Kreisbahngeschwindigkeit `MOON_CIRCULAR_VELOCITY`.
Dadurch bleibt seine Umlaufbahn vollständig in der X/Y-Ebene, was die Visualisierung
stark vereinfacht.

Als Zeitschritt verwenden wir `DEFAULT_EARTH_MOON_TIME_STEP = 1 h`. Ein realer
Mondumlauf dauert ca. 27 Tage — ein deutlich feinerer Zeitschritt würde die
Simulation unnötig verlangsamen, ohne die Genauigkeit spürbar zu verbessern.

**Recherchierte Referenzwerte:**

| Größe | Wert | Quelle |
|---|---|---|
| Erdmasse | 5.972 × 10²⁴ kg | NASA Earth Fact Sheet |
| Erdradius | 6.371 × 10⁶ m | NASA Earth Fact Sheet |
| Mondmasse | 7.346 × 10²² kg | NASA Moon Fact Sheet |
| Mondradius | 1.7374 × 10⁶ m | NASA Moon Fact Sheet |
| Erde-Mond-Abstand | 3.844 × 10⁸ m | NASA Moon Fact Sheet |
| Siderische Umlaufzeit | 27.322 Tage | NASA Moon Fact Sheet |

Die folgende Zelle simuliert 30 Tage — das entspricht etwas mehr als einer vollen
Mondumlaufperiode und eignet sich damit gut zur visuellen Kontrolle:

### Der Code dahinter — die Startaufstellung

Die Startkörper kommen aus `create_earth_moon`: Erde ruht im Ursprung, Mond auf der X-Achse mit
Kreisbahngeschwindigkeit in +Y.

```python
# src/scenarios.py
def create_earth_moon():
    earth = Body("Earth", constants.EARTH_MASS, constants.EARTH_RADIUS, [0,0,0], [0,0,0])
    moon  = Body("Moon",  constants.MOON_MASS,  constants.MOON_RADIUS,
                 [constants.MOON_START_X, 0, 0], [0, constants.MOON_CIRCULAR_VELOCITY, 0])
    config = {"time_step": constants.DEFAULT_EARTH_MOON_TIME_STEP}
    return [earth, moon], config
```

- Jeder Körper ist ein `Body` mit Masse, Radius, Position (x/y/z), Geschwindigkeit (x/y/z) —
  genau die geforderten Eigenschaften.
- Erde in `[0,0,0]`, ruhend. Mond bei `MOON_START_X` auf der X-Achse, Geschwindigkeit in +Y
  (`MOON_CIRCULAR_VELOCITY`, Def 1.5) → Bahn bleibt in der X/Y-Ebene.
- Rückgabe ist immer `(bodies, config)`. Im `config` steht der konfigurierbare Zeitschritt —
  so behandelt die Simulation jedes Szenario gleich.

Wie aus diesen Startkörpern echte Bewegung wird, zeigt gleich der Simulation-Motor.

### Der Motor — `simulation.py`

`Simulation` steckt alles zusammen: sie nimmt die Startkörper, geht Schritt für Schritt duch die Simulation und
protokolliert jeden Schritt in der History.

**Setup — Zeitschritt, Dauer, Schrittzahl:**

```python
# src/simulation.py
def __init__(self, bodies, config, duration, integrator=None):
    self.bodies = bodies
    self.dt = config["time_step"]
    self.duration = duration*24*60*60 # Convert duration from days to seconds
    self.steps = round(self.duration/self.dt)
    self.integrator = integrator if integrator is not None else Verlet()
    self.t = 0.0
    self.history = []
    self._save_snapshot()
```

- `dt` = konfigurierbarer Zeitschritt aus dem Szenario (1 h beim Mond, 10 s bei der Kanone).
- `duration` wird von Tagen in Sekunden umgerechnet, `steps = round(duration / dt)` ist die
  Anzahl Integrationsschritte. Das ist der beschleunigte Ablauf: 30 Tage Mondbahn rechnen wir
  in Sekunden, statt real 4 Wochen zu warten.
- Ohne expliziten Integrator wird Verlet genutzt. Der Ausgangszustand wird sofort als erster
  Snapshot gespeichert.

**Szenario-Auswahl:**

```python
# src/simulation.py
@staticmethod
def build_simulation(cannon, duration, integrator=None, cannonball_angle=None, cannonball_speed=None):
    if cannon:
        if cannonball_angle is None or cannonball_speed is None:
            raise ValueError("To start the cannon shot, provide the cannon angle and the cannonball speed")
        bodies, config = create_cannon_shot(cannonball_speed, cannonball_angle)
    elif duration == 67:
        bodies, config = create_67()
    else:
        bodies, config = create_earth_moon()
    return Simulation(bodies, config, duration=duration, integrator=integrator)
```

- Wählt anhand `cannon` das passende Szenario und baut die `Simulation`.
- Der `raise ValueError(...)` ist unser erstes Fehlerbehandlungs-Beispiel.

**Der Zeitschritt-Loop + die History:**

```python
# src/simulation.py
def step(self):
    self.integrator.step(self.bodies, self.dt)
    self.bodies = Collisions.handle(self.bodies, self.dt)
    self.t += self.dt
    self._save_snapshot()

def _save_snapshot(self):
    snapshot = {
        "t": self.t,
        "bodies": [
            {
            "name": b.name,
            "mass": b.mass,
            "radius": b.radius,
            "position": b.position.copy(),
            "velocity": b.velocity.copy()
            }
            for b in self.bodies
        ]
    }
    self.history.append(snapshot)

def simulate(self):
    for i in range(self.steps):
        self.step()
```

- **Ein Schritt** = integrieren (Integrator) → Kollisionen prüfen (`Collisions.handle`) → Zeit
  erhöhen → Snapshot speichern.
- **Snapshot** = `{"t": Zeit, "bodies": [{name, mass, radius, position, velocity}, …]}`. `position`
  und `velocity` werden kopiert (`.copy()`), damit spätere Schritte alte Snapshots nicht
  rückwirkend verändern.
- `simulate()` ruft `step()` genau `steps`-mal auf — fertig ist die komplette History, aus der
  Visualisierung und Analyse alles ableiten.

### Der ganze Datenfluss in einem Aufruf — `run` & `render`

Alles, was ihr bisher gesehen habt, steckt in fünf Zeilen. Das ruft auch das Panel im
Hintergrund auf.

```python
# src/panel.py
def run(cannon, integrator_name, duration, angle, speed, size_factor):
    integrator = map_integrator(integrator_name)
    sim = Simulation.build_simulation(cannon, duration, integrator=integrator,
                                      cannonball_angle=angle, cannonball_speed=speed)
    sim.simulate()
    viz = Visualization()
    viz.animate(sim.history, size_factor=size_factor)
    return viz


def render(viz):
    html = viz.anim.to_jshtml()
    plt.close(viz.fig)
    return HTML(html)
```

1. `map_integrator` wählt das Verfahren (`"Verlet"` → Störmer-Verlet).
2. `build_simulation` baut über `scenarios` die Startaufstellung.
3. `sim.simulate()` schrittet durch die Zeit und füllt die History.
4. `Visualization().animate(...)` macht daraus die Animation.
5. `return viz`.

`render` verwandelt die Animation in HTML (`to_jshtml`), schließt die Figur (kein doppeltes
Standbild) und gibt anzeigbares HTML zurück. Die nächste Zelle ruft genau `run(False, …)` für
30 Tage Erde-Mond auf.

### Fehlerbehandlung — aussagekräftige Meldungen auf drei Ebenen

Wir prüfen Fehler dort, wo sie entstehen, mit klaren Meldungen.

**1. Eingaben validieren — `Body.__init__`:** ein Körper mit unsinnigen Werten darf gar nicht erst existieren.

```python
# src/body.py
def __init__(self, name, mass, radius, position, velocity):

    if mass <= 0:
        raise ValueError(f"mass must be positive, got {mass}")
    if radius <= 0:
        raise ValueError(f"radius must be positive, got {radius}")

    self.name = name
    self.mass = mass
    self.radius = radius

    self.position = np.array(position, dtype=float)
    self.velocity = np.array(velocity, dtype=float)

    if self.position.shape != (3,):
        raise ValueError(f"position must have 3 values [x, y, z], got {position}")
    if self.velocity.shape != (3,):
        raise ValueError(f"velocity must have 3 values [vx, vy, vz], got {velocity}")

    self.position_previous = None
    self.acceleration = np.array([0.0, 0.0, 0.0])
```

**2. Vorbedingungen prüfen** — `build_simulation` (Kanone ohne Winkel/Speed) und `map_integrator`
(unbekannter Integrator) werfen sofort mit klarer Meldung:

```python
# src/panel.py
def map_integrator(name):
    if name == "Verlet":
        return Verlet()
    elif name == "Euler":
        return Euler()
    else:
        raise ValueError(f"Input: {name}, Expected: Verlet or Euler")
```

**3. Im UI abfangen** — der Start-Knopf des Panels fängt jede Exception und zeigt sie als Text,
statt die Zelle abstürzen zu lassen:

```python
# src/panel.py  (Ausschnitt aus build_panel)
try:
    viz = run(cannonball_checkbox.value, integrator.value, duration.value,
              cannonball_angle.value, cannonball_speed.value * 1000, scale.value)
    html = render(viz)
    output.clear_output()
    display(html)
except Exception as e:
    print(f"Fehler: {e}")
```

In [ ]:
emviz = run(False, "Verlet", 30, None, None, 15000)
html = render(emviz)
display(html)

## 3. Teil 2.1: Die Columbiade (Kanonenschuss): Gibt es einen Treffer?

### Schüsse mit verschiedenen Winkeln und Geschwindigkeiten

- 7km/s     0°
- 9km/s     0°
- 12km/s    0°
- 7km/s     15°
- 9km/s     15°
- 12km/s    15°
- 7km/s     30°
- 9km/s     30°
- 12km/s    30°

Es werden alle 9 Simulationen berechnet, eine Geschwindigkeit von 7 oder 9 km/s ist nicht ausreichend, um den Mond zu treffen, da die Gravitation der Erde stärker ist und die Kanonenkugeln zurück auf die Erde fallen. Erst bei 12 km/s ist die Geschwindigkeit stark genug, jedoch trifft keiner der Winkel.

### Die Abschussgeometrie — `create_cannon_shot`

Gleiche Erde-Mond-Aufstellung, plus ein **Projektil** an der Erdoberfläche. Winkel und
Geschwindigkeit bestimmen Startposition und -richtung.

```python
# src/scenarios.py
def create_cannon_shot(speed, angle_deg):
    angle_rad = math.radians(angle_deg)
    earth = Body("Earth", constants.EARTH_MASS, constants.EARTH_RADIUS, [0,0,0], [0,0,0])
    moon  = Body("Moon",  constants.MOON_MASS,  constants.MOON_RADIUS,
                 [constants.MOON_START_X, 0, 0], [0, constants.MOON_CIRCULAR_VELOCITY, 0])
    start_r = constants.EARTH_RADIUS + constants.PROJECTILE_RADIUS + 1000.0
    pos_x = start_r * math.cos(angle_rad)
    pos_y = start_r * math.sin(angle_rad)
    vel_x = speed * math.cos(angle_rad)
    vel_y = speed * math.sin(angle_rad)
    projectile = Body("Projectile", constants.PROJECTILE_MASS, constants.PROJECTILE_RADIUS,
                      [pos_x, pos_y, 0], [vel_x, vel_y, 0])
    config = {"time_step": constants.DEFAULT_CANNON_TIME_STEP}
    return [earth, moon, projectile], config
```

- Der Winkel dreht Startposition und Startrichtung.
- `start_r` setzt das Projektil knapp über die Erdoberfläche (nicht sofort „Erd-Kollision").
- Zeitschritt viel feiner (10 s statt 1 h) — ein Geschoss ist schnell.

### Definition 1.6 im Code — Kollision & inelastischer Merge

Ein Mondtreffer ist genau dieses Ereignis: Mond und Projektil berühren sich und verschmelzen.

```python
# src/collisions.py
class Collisions:

    @staticmethod
    def handle(bodies, dt):
        for i in range(len(bodies)):
            for j in range(i + 1, len(bodies)):
                b1 = bodies[i]
                b2 = bodies[j]
                distance = np.linalg.norm(b1.position - b2.position)

                if distance <= b1.radius + b2.radius:
                    new_body = Collisions.merge(b1, b2, dt)
                    # remove the two old bodies, add the new one,
                    # and start from the beginning till nothing changes
                    new_list = [b for k, b in enumerate(bodies)
                                if k != i and k != j]
                    new_list.append(new_body)
                    return Collisions.handle(new_list, dt)
        return bodies

    @staticmethod
    def merge(b1, b2, dt):
        name_new = b1.name + "_" + b2.name
        m_new = b1.mass + b2.mass
        v_new = (b1.mass * b1.velocity + b2.mass * b2.velocity) / m_new
        r_new = (b1.mass * b1.position + b2.mass * b2.position) / m_new
        radius_new = (b1.radius**3 + b2.radius**3) ** (1 / 3)

        new_body = Body(name=name_new, mass=m_new, position=r_new,
                        velocity=v_new, radius=radius_new)
        # set position_previous for verlet calculations
        new_body.position_previous = r_new - v_new * dt
        return new_body
```

- **Berührung** = Abstand ≤ Summe der Radien.
- **Rekursiver Restart:** nach jedem Merge wird `handle` neu gestartet, bis nichts mehr kollidiert.
- **Impulserhaltung (Def 1.6):** `v_new = (m₁·v₁ + m₂·v₂)/(m₁+m₂)`, Masse addiert sich, Position = Massenschwerpunkt.
- **Radius:** `(r₁³ + r₂³)^(1/3)` — die Volumina addieren sich (Square-Cube).
- **Namensschema:** der neue Körper heißt `"Moon_Projectile"` — daran erkennt die Analyse den Treffer.

In [ ]:
speeds = [7, 9, 12]
angles = [0, 15, 30]

for speed in speeds:
    for angle in angles:
        display(HTML(f"<h3>Geschwindigkeit: {speed} km/s | Winkel: {angle}°</h3>"))
        try:
            viz = run(True, "Verlet", 1, angle, speed * 1000, 15000)
            html = render(viz)
            display(html)
        except Exception as e:
            print(f"Fehler in Winkel {angle} und Geschwindigkeit {speed}: {e}")

### Simulation mit variablen Werten

### Von der History zum Bild — `visualization.py`

Die Visualisierung liest die History und macht daraus die Animation.

**History → Frames:**

```python
# src/visualization.py
def data_adapter(self, history):
    timestamps = np.array([snap["t"] for snap in history])
    bodies_at_frame = [snap["bodies"] for snap in history]
    return timestamps, bodies_at_frame
```

**Anpassbarer Maßstab (nichtlinear):** Erde und Projektil sind in Wirklichkeit extrem
unterschiedlich groß. Damit das kleine Projektil sichtbar bleibt, skalieren wir den Anzeigeradius
nichtlinear über die Wurzel:

```python
# src/visualization.py
def _scale_radius(real_radius, mode, size_factor):
    if mode == "linear":
        return float(real_radius * size_factor)
    elif mode == "sqrt":
        return float(np.sqrt(real_radius) * size_factor)
    else:
        raise ValueError(f"Unknown mode: {mode}")
```

- Wir zeichnen mit `mode="sqrt"` → dämpft riesige Größenunterschiede.
- `size_factor` ist der anpassbare Maßstab — im Panel der Regler „Körpergröße". Damit ist
  die Anforderung „Maßstab automatisch oder interaktiv anpassbar" erfüllt.

**Treffer sichtbar machen:** Die Kollision erkennt die Visualisierung daran, dass zwischen zwei
Frames ein Körpername verschwindet (weil zwei zu `"Moon_Projectile"` verschmolzen sind), und
setzt dort ein rotes **X**.

```python
# src/visualization.py
def find_collisions(self, bodies_at_frame):
    # a collision = names vanish between two frames
    events = []
    for i in range (1, len(bodies_at_frame)):
        prev_names = {b["name"] for b in bodies_at_frame[i - 1]}
        cur_names = {b["name"] for b in bodies_at_frame[i]}

        vanished = prev_names - cur_names
        if not vanished:
            continue

        vanished_bodies = [b for b in bodies_at_frame[i - 1]
                           if b["name"] in vanished]
        if vanished_bodies:
            smallest = min(vanished_bodies, key=lambda b: b["mass"])
            pts = [(smallest["position"][0], smallest["position"][1])]
        else:
            pts = []

        x = float(np.mean([p[0] for p in pts]))
        y = float(np.mean([p[1] for p in pts]))
        events.append((i, (x, y)))
    return events
```

Jetzt live: Panel öffnen → die Standardwerte (12,3 km/s, 10°) sind ein Treffer → „Start"
drücken → das Projektil trifft den Mond, die beiden verschmelzen, das rote X erscheint.

In [ ]:
panel, refs = build_panel()
display(panel)

### Wann wird der Mond getroffen?

### Die Forschungsfrage — Trefferlogik & der schnelle Sweep

**Was zählt als Treffer?** Im vollen Lauf ist es die Kollision (Def 1.6): Mond + Projektil werden
`"Moon_Projectile"`. Genau darauf prüft die Analyse:

```python
# src/analysis.py
def _is_moon_hit(history):
    # after a merge the body is named "Moon_Projectile"
    for snapshot in history:
        for body in snapshot["bodies"]:
            name = body["name"]
            if "Projectile" in name and "Moon" in name:
                return True
    return False
```

`min_distance_to_moon` misst zusätzlich, wie nah das Projektil dem Mond kam — das färbt die
Heatmap (auch knappe Fehlschüsse werden sichtbar).

**Der schnelle Weg:** `sweep_fast` schießt alle ~2000 Kombinationen gleichzeitig durch eine
einmal vorberechnete Erde-Mond-Bahn — vektorisiert mit NumPy, daher „~8 s".

```python
# src/analysis.py
def sweep_fast(speeds, angles, duration=3.5, dt=None):
    # fast vectorized sweep (what the notebook calls). same grid output as sweep().
    speeds = np.asarray(speeds, dtype=float)
    angles = np.asarray(angles, dtype=float)
    if dt is None:
        dt = constants.DEFAULT_CANNON_TIME_STEP

    earth_xyz, moon_xyz = _earth_moon_trajectory(duration, dt)

    start_r = constants.EARTH_RADIUS + constants.PROJECTILE_RADIUS + 1000.0
    speed_grid = np.repeat(speeds, len(angles))
    angle_grid = np.tile(angles, len(speeds))
    angle_rad = np.radians(angle_grid)

    pos0 = np.zeros((speed_grid.size, 3))
    vel0 = np.zeros((speed_grid.size, 3))
    pos0[:, 0] = start_r * np.cos(angle_rad)
    pos0[:, 1] = start_r * np.sin(angle_rad)
    vel0[:, 0] = speed_grid * np.cos(angle_rad)
    vel0[:, 1] = speed_grid * np.sin(angle_rad)

    min_moon, moon_hit = _propagate_projectiles(pos0, vel0, earth_xyz, moon_xyz, dt)

    shape = (len(speeds), len(angles))
    return {"speeds": speeds, "angles": angles,
            "hits": moon_hit.reshape(shape), "min_dist": min_moon.reshape(shape)}
```

- **Einmal die Bahn, dann alle Schüsse:** Erde-/Mondpositionen werden einmal vorberechnet,
  danach fliegen alle Projektile durch dieses Feld — der ganze Geschwindigkeits-Trick.
- Gleiche Startgeometrie wie `create_cannon_shot`, nur für alle Paare gleichzeitig (`np.repeat`/`np.tile`).
- Ergebnis = Gitter aus kleinster Monddistanz + Treffer-Flag → das färbt die Heatmap.

Die Heatmap zeigt: kein isolierter Punkt, sondern ein Korridor (schneller → flacherer
Vorhaltewinkel), links hart begrenzt durch die Fluchtgeschwindigkeit (Def 1.4). Das ist unsere
Antwort auf die Forschungsfrage.

In [ ]:
analyse_speeds = np.arange(10.5, 13.01, 0.05) * 1000   # 51 speeds
analyse_winkel = np.arange(2.0, 22.01, 0.5)            # 41 angles

korridor = sweep_fast(analyse_speeds, analyse_winkel)   # ~8 s for ~2000 cells
ax = plot_corridor_heatmap(korridor)

## 4. Tests

Pro Modul gibt es eine Testdatei im(`tests/`) Ordner. Getestet werden u. a. die Gravitationsberechnung, der
Verlet-Schritt (Abstand nimmt über mehrere Schritte ab), Kollision/Merge (Massen- und
Impulserhaltung), die Szenario-Startwerte und die Konstanten. Die nächste Zelle findet und startet
alle Tests automatisch per `unittest`-Discovery.

In [ ]:
# execute all tests
loader = unittest.TestLoader()
suite = loader.discover(start_dir="tests", pattern="test_*.py")
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 5. Zusammenfassung und Quellen

- NASA Earth Fact Sheet: https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html
- NASA Moon Fact Sheet: https://nssdc.gsfc.nasa.gov/planetary/factsheet/moonfact.html
- Leifiphysik Gravitation: https://www.leifiphysik.de/mechanik/gravitationsgesetz-und-feld/grundwissen/gravitationsgesetz-von-newton
- Gravitation: https://en.wikipedia.org/wiki/Newton's_law_of_universal_gravitation
- Verlet: https://de.wikipedia.org/wiki/Verlet-Algorithmus
- Verlet: https://www.algorithm-archive.org/contents/verlet_integration/verlet_integration.html
- Inelastische Kollision: https://www.leifiphysik.de/mechanik/impulserhaltung-und-stoesse/grundwissen/zentraler-vollkommen-unelastischer-stoss